# Setup: `v2_pred_patch` Grid Index Update Benchmark

## Overview

This notebook documents the environment setup for the **Update grid indices v2_pred_patch** benchmark.
It does **not** drop, truncate, or modify the existing `v2_pred_patch` table.

## Schema Reference

The `v2_pred_patch` table (`db_technical_design.md` §1.4) has the following relevant columns:

| Column        | Type        | Description                                        |
|---------------|-------------|----------------------------------------------------|
| `id`          | INTEGER PK  | Row identifier                                     |
| `patch_uid`   | BIGINT      | Unique patch identifier                            |
| `embed_coords`| POINT       | Embedding coordinates (x, y)                       |
| `grid_cell_i` | INTEGER     | Grid index I derived from embed_coords (IJ scheme) |
| `grid_cell_j` | INTEGER     | Grid index J derived from embed_coords (IJ scheme) |
| `event_ts`    | TIMESTAMPTZ | Time of last update                                |
| `pred_label`  | INTEGER     | Predicted label class                              |
| `patch_coords`| POINT       | Patch coordinates within the source image          |

**Indexes on `v2_pred_patch`:**
- `v2_pred_patch_pkey` — B-tree on `id` (primary key)
- `idx_v2_pred_patch_grid_cells` — B-tree on `(grid_cell_i, grid_cell_j)`

**Table size at benchmark time**: ~800 million rows, ~106 GB total (75 GB heap + 31 GB indexes)

## Benchmark Strategy

- **No structural changes to `v2_pred_patch`** — the production table is used as-is.
- A temporary **staging table** `bench_v2_pred_patch_temp` is created to hold the original
  `grid_cell_i` / `grid_cell_j` values for the 1000 target rows, enabling safe teardown.
- The **bloat scenario** is implemented by performing repeated update rounds without VACUUM to
  accumulate dead tuples on the target pages, then measuring the update performance penalty.
- The 1000 rows to update are selected **randomly** using `TABLESAMPLE SYSTEM(0.002)` to obtain
  a page-level random sample (~5ms on 800M rows), then trimmed to exactly 1000. This simulates
  a realistic DL worker workload where patches being updated are not necessarily sequential.

## Connection
Reads from environment variables `DB_HOST`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`;
falls back to the prototyping defaults if not set.

## Important
- **NEVER** runs DROP, TRUNCATE, or DELETE on `v2_pred_patch`.
- After the benchmark, all original `grid_cell_i` / `grid_cell_j` values are **restored**.
- The temporary staging table `bench_v2_pred_patch_temp` is dropped on teardown.

In [ ]:
import os
import time
import random
import psycopg2
from psycopg2.extras import execute_values

# ---------------------------------------------------------------------------
# Connection parameters — read from env vars, fall back to prototyping defaults
# ---------------------------------------------------------------------------
DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

BENCH_ROWS = 1_000        # Number of rows targeted in each timed trial

conn = psycopg2.connect(DSN)
conn.autocommit = True
cur = conn.cursor()

# Report PG version and table stats
cur.execute('SELECT version();')
pg_ver = cur.fetchone()[0].split(',')[0]
print(f'Connected to {pg_ver}')

cur.execute('SELECT COUNT(*) FROM v2_pred_patch;')
row_count = cur.fetchone()[0]
print(f'v2_pred_patch row count: {row_count:,}')

cur.execute("""
    SELECT
      pg_size_pretty(pg_total_relation_size('v2_pred_patch')) AS total_size,
      pg_size_pretty(pg_relation_size('v2_pred_patch'))       AS heap_size,
      pg_size_pretty(pg_indexes_size('v2_pred_patch'))        AS index_size
""")
sizes = cur.fetchone()
print(f'Table sizes: total={sizes[0]}, heap={sizes[1]}, indexes={sizes[2]}')

# ---------------------------------------------------------------------------
# Randomly select 1000 IDs using TABLESAMPLE SYSTEM (page-level random sampling)
# Oversample to 1200 to ensure we always get at least 1000 unique IDs
# ---------------------------------------------------------------------------
print('\n--- Randomly selecting 1000 IDs using TABLESAMPLE SYSTEM(0.002) ---')
t0 = time.perf_counter()
cur.execute('SELECT id FROM v2_pred_patch TABLESAMPLE SYSTEM(0.002) LIMIT 1200;')
sample_rows = cur.fetchall()
elapsed_sample = time.perf_counter() - t0
candidate_ids = [r[0] for r in sample_rows]
random.shuffle(candidate_ids)     # shuffle to remove any page-ordering bias
TARGET_IDS = candidate_ids[:BENCH_ROWS]
print(f'Randomly selected {len(TARGET_IDS)} IDs in {elapsed_sample*1000:.1f}ms')
print(f'ID spread: min={min(TARGET_IDS)}, max={max(TARGET_IDS)}')
print(f'Unique IDs: {len(set(TARGET_IDS))}')

# Read and snapshot the original grid values for the 1000 target rows
cur.execute("""
    SELECT id, grid_cell_i, grid_cell_j
    FROM v2_pred_patch
    WHERE id = ANY(%s)
    ORDER BY id;
""", (TARGET_IDS,))
original_values = cur.fetchall()
print(f'\nSnapshotted {len(original_values)} rows for restore after benchmark.')
print(f'Sample: {original_values[:3]}')

# Create staging table to hold original values for safe teardown
cur.execute('DROP TABLE IF EXISTS bench_v2_pred_patch_temp;')
cur.execute("""
    CREATE UNLOGGED TABLE bench_v2_pred_patch_temp (
        id          INTEGER PRIMARY KEY,
        grid_cell_i INTEGER,
        grid_cell_j INTEGER
    );
""")
execute_values(
    cur,
    'INSERT INTO bench_v2_pred_patch_temp (id, grid_cell_i, grid_cell_j) VALUES %s;',
    original_values
)
print(f'Staging table bench_v2_pred_patch_temp created with {len(original_values)} rows.')

conn.close()
print('Setup complete.')

Connected to PostgreSQL 15.17
v2_pred_patch row count: 800,003,200
Table sizes: total=106 GB, heap=75 GB, indexes=31 GB

--- Randomly selecting 1000 IDs using TABLESAMPLE SYSTEM(0.002) ---
Randomly selected 1000 IDs in 5.4ms
ID spread: min=2701377, max=67657948
Unique IDs: 1000

Snapshotted 1000 rows for restore after benchmark.
Sample: [(2701377, 1119, 2648), (2701379, 1277, 3559), (2701383, 1034, 2139)]
Staging table bench_v2_pred_patch_temp created with 1000 rows.
Setup complete.


In [ ]:
# ---------------------------------------------------------------------------
# TEARDOWN — run this cell AFTER the benchmark to restore original data
# and clean up the staging table.
# This cell is also inlined at the end of the benchmark notebook.
# ---------------------------------------------------------------------------
import os
import psycopg2

DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

conn = psycopg2.connect(DSN)
conn.autocommit = True
cur = conn.cursor()

cur.execute("SELECT EXISTS (SELECT 1 FROM pg_tables WHERE tablename = 'bench_v2_pred_patch_temp');")
staging_exists = cur.fetchone()[0]

if staging_exists:
    # Restore original grid_cell_i / grid_cell_j values from staging table
    cur.execute("""
        UPDATE v2_pred_patch AS target
        SET
            grid_cell_i = staging.grid_cell_i,
            grid_cell_j = staging.grid_cell_j
        FROM bench_v2_pred_patch_temp AS staging
        WHERE target.id = staging.id;
    """)
    print(f'Restored original grid_cell_i/j for {cur.rowcount} rows in v2_pred_patch.')

    cur.execute('DROP TABLE IF EXISTS bench_v2_pred_patch_temp;')
    print('Staging table bench_v2_pred_patch_temp dropped.')
else:
    print('Staging table not found — may already have been cleaned up.')

# Verify restoration using known target IDs (re-run setup cell first if TARGET_IDS is not defined)
if 'TARGET_IDS' in dir():
    sample_ids = sorted(TARGET_IDS)[:3]
    cur.execute('SELECT id, grid_cell_i, grid_cell_j FROM v2_pred_patch WHERE id = ANY(%s) ORDER BY id;', (sample_ids,))
    print('Sample restored rows:', cur.fetchall())

conn.close()
print('Teardown complete.')

Restored original grid_cell_i/j for 1000 rows in v2_pred_patch.
Staging table bench_v2_pred_patch_temp dropped.
Sample restored rows: [(2701377, 1119, 2648), (2701379, 1277, 3559), (2701383, 1034, 2139)]
Teardown complete.
